In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label'
]
df_train = pd.read_csv("nsl_kdd_train.csv", header=None, names=columns)
df_test = pd.read_csv("nsl_kdd_test.csv", header=None, names=columns)


In [2]:
# قاموس (dictionary) بيربط كل نوع هجوم تفصيلي بفئته الأكبر
attack_mapping = {
    'normal': 'normal',
    # DoS - Denial of Service: هجمات تهدف تعطيل الخدمة (إغراق بالطلبات)
    'neptune': 'DoS', 'smurf': 'DoS', 'back': 'DoS', 'teardrop': 'DoS',
    'pod': 'DoS', 'land': 'DoS',
    # Probe - استطلاع/مسح الشبكة قبل الهجوم (زي nmap يلي أنت مستخدمه بالـ CTF)
    'satan': 'Probe', 'ipsweep': 'Probe', 'portsweep': 'Probe', 'nmap': 'Probe',
    # R2L - Remote to Local: وصول غير مصرح من جهاز بعيد
    'warezclient': 'R2L', 'guess_passwd': 'R2L', 'warezmaster': 'R2L',
    'imap': 'R2L', 'ftp_write': 'R2L', 'multihop': 'R2L', 'phf': 'R2L',
    'spy': 'R2L',
    # U2R - User to Root: تصعيد صلاحيات لـ root (زي شغلك بـ cron hijacking بـ Printer Shares!)
    'buffer_overflow': 'U2R', 'rootkit': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R'
}
# .map() بتستبدل كل قيمة بعمود label بالقيمة المقابلة إلها بالقاموس
df_train['attack_category'] = df_train['label'].map(attack_mapping)
df_test['attack_category'] = df_test['label'].map(attack_mapping)

print("توزيع Train:\n", df_train['attack_category'].value_counts(dropna=False))
print("\nتوزيع Test:\n", df_test['attack_category'].value_counts(dropna=False))

توزيع Train:
 attack_category
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

توزيع Test:
 attack_category
normal    9711
DoS       5741
NaN       3750
R2L       2199
Probe     1106
U2R         37
Name: count, dtype: int64


In [3]:
# fillna() بتستبدل أي قيمة NaN بقيمة محددة
# هون منحط "Unknown" لأي نوع هجوم بالـ Test مش موجود أصلًا بالـ Train
df_test['attack_category'] = df_test['attack_category'].fillna('Unknown')

print(df_test['attack_category'].value_counts())

attack_category
normal     9711
DoS        5741
Unknown    3750
R2L        2199
Probe      1106
U2R          37
Name: count, dtype: int64


In [4]:
X_train = df_train.drop(['label', 'attack_category'], axis=1)
y_train = df_train['attack_category']
X_test = df_test.drop(['label', 'attack_category'], axis=1)
y_test = df_test['attack_category']
categorical_cols = ['protocol_type', 'service', 'flag']

preprocessor = ColumnTransformer([
    ("encoder", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")   # باقي الأعمدة (الرقمية) تمر كما هي بدون تغيير

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

In [5]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\n--- تقرير التصنيف التفصيلي ---")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.7229

--- تقرير التصنيف التفصيلي ---
              precision    recall  f1-score   support

         DoS       0.95      0.97      0.96      5741
       Probe       0.62      1.00      0.77      1106
         R2L       0.98      0.06      0.12      2199
         U2R       0.25      0.05      0.09        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.64      0.97      0.77      9711

    accuracy                           0.72     22544
   macro avg       0.57      0.51      0.45     22544
weighted avg       0.65      0.72      0.63     22544



In [6]:
from sklearn.ensemble import RandomForestClassifier

pipeline_balanced = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'   # ← الإضافة الوحيدة
    ))
])

pipeline_balanced.fit(X_train, y_train)
y_pred_balanced = pipeline_balanced.predict(X_test)

acc_balanced = accuracy_score(y_test, y_pred_balanced)
print(f"Accuracy (balanced): {acc_balanced:.4f}")
print("\n--- تقرير التصنيف بعد الموازنة ---")
print(classification_report(y_test, y_pred_balanced, zero_division=0))

Accuracy (balanced): 0.7213

--- تقرير التصنيف بعد الموازنة ---
              precision    recall  f1-score   support

         DoS       0.95      0.99      0.97      5741
       Probe       0.66      1.00      0.80      1106
         R2L       0.76      0.01      0.01      2199
         U2R       0.33      0.03      0.05        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.64      0.97      0.77      9711

    accuracy                           0.72     22544
   macro avg       0.56      0.50      0.43     22544
weighted avg       0.62      0.72      0.62     22544



In [7]:
pipeline_balanced_sub = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample'   # بدل 'balanced'
    ))
])

pipeline_balanced_sub.fit(X_train, y_train)
y_pred_sub = pipeline_balanced_sub.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_sub):.4f}")
print(classification_report(y_test, y_pred_sub, zero_division=0))

Accuracy: 0.7217
              precision    recall  f1-score   support

         DoS       0.95      0.99      0.97      5741
       Probe       0.66      1.00      0.80      1106
         R2L       0.75      0.00      0.01      2199
         U2R       0.25      0.03      0.05        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.64      0.97      0.77      9711

    accuracy                           0.72     22544
   macro avg       0.54      0.50      0.43     22544
weighted avg       0.62      0.72      0.62     22544



In [8]:

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # مختلفة عن sklearn.pipeline.Pipeline

# ----------------------------------------------------
# نفس الـ preprocessor يلي عندك (ColumnTransformer بالـ OneHotEncoder)
# بس هلق منضيف خطوة SMOTE بين المعالجة والموديل
pipeline_smote = ImbPipeline([
    ("preprocessing", preprocessor),
    ("smote", SMOTE(random_state=42, k_neighbors=5)),  # k_neighbors=5 افتراضي، كافي لأنه U2R عندها 52 صف بالتدريب
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

# ----------------------------------------------------
# .fit() هون بيصير بالترتيب الصحيح تلقائيًا:
# 1) الـ Encoder بيتعلم من X_train ويحوله لأرقام
# 2) SMOTE بيولّد صفوف اصطناعية جديدة بس لبيانات التدريب المُحوّلة
# 3) الموديل بيتدرب على الداتا الموسّعة (حقيقية + اصطناعية)
pipeline_smote.fit(X_train, y_train)

# ----------------------------------------------------
# .predict() هون بيطبق بس الـ Encoder (مش SMOTE - وهاد تلقائي بفضل imblearn.Pipeline)
# على X_test الحقيقي بدون أي تعديل - هيك التقييم نظيف 100%
y_pred_smote = pipeline_smote.predict(X_test)

acc_smote = accuracy_score(y_test, y_pred_smote)
print(f"Accuracy (SMOTE): {acc_smote:.4f}")
print("\n--- تقرير التصنيف بعد SMOTE ---")
print(classification_report(y_test, y_pred_smote, zero_division=0))

c:\Users\Baraa Hereh\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Accuracy (SMOTE): 0.7224

--- تقرير التصنيف بعد SMOTE ---
              precision    recall  f1-score   support

         DoS       0.92      0.99      0.95      5741
       Probe       0.62      1.00      0.77      1106
         R2L       0.68      0.01      0.02      2199
         U2R       0.31      0.14      0.19        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.65      0.97      0.78      9711

    accuracy                           0.72     22544
   macro avg       0.53      0.52      0.45     22544
weighted avg       0.61      0.72      0.62     22544



In [9]:
from imblearn.over_sampling import SMOTENC

# ----------------------------------------------------
# SMOTENC لازم يعرف مسبقًا مين الأعمدة الفئوية (بمواقعها/index) قبل الـ Encoding
# هون المشكلة: SMOTENC بده يشتغل قبل One-Hot Encoding، مش بعده
# فلازم نغيّر ترتيب الـ Pipeline: SMOTENC أول، بعدين Encoding، بعدين الموديل

categorical_features_idx = [X_train.columns.get_loc(c) for c in ['protocol_type', 'service', 'flag']]

pipeline_smotenc = ImbPipeline([
    ("smotenc", SMOTENC(categorical_features=categorical_features_idx, random_state=42, k_neighbors=5)),
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline_smotenc.fit(X_train, y_train)
y_pred_smotenc = pipeline_smotenc.predict(X_test)

print(f"Accuracy (SMOTENC): {accuracy_score(y_test, y_pred_smotenc):.4f}")
print(classification_report(y_test, y_pred_smotenc, zero_division=0))

c:\Users\Baraa Hereh\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\Baraa Hereh\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Accuracy (SMOTENC): 0.7323
              precision    recall  f1-score   support

         DoS       0.90      0.99      0.94      5741
       Probe       0.65      1.00      0.79      1106
         R2L       0.96      0.12      0.21      2199
         U2R       0.27      0.08      0.12        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.66      0.97      0.79      9711

    accuracy                           0.73     22544
   macro avg       0.57      0.53      0.47     22544
weighted avg       0.64      0.73      0.64     22544



In [10]:
# نشوف أنواع R2L الفرعية الموجودة بكل من Train و Test لحالها
train_r2l_types = set(df_train[df_train['attack_category'] == 'R2L']['label'].unique())
test_r2l_types = set(df_test[df_test['attack_category'] == 'R2L']['label'].unique())

print("أنواع R2L بالـ Train:", train_r2l_types)
print("أنواع R2L بالـ Test:", test_r2l_types)
print("\nأنواع موجودة بالـ Test وغير موجودة بالـ Train:", test_r2l_types - train_r2l_types)

# كمان نشوف كم صف فعليًا من هاي الأنواع "الجديدة" مقارنة بالكل
new_types = test_r2l_types - train_r2l_types
count_new = df_test[df_test['label'].isin(new_types)].shape[0]
print(f"\nعدد صفوف R2L من أنواع جديدة كليًا: {count_new} من أصل {df_test[df_test['attack_category']=='R2L'].shape[0]}")

أنواع R2L بالـ Train: {'ftp_write', 'imap', 'multihop', 'warezmaster', 'spy', 'guess_passwd', 'warezclient', 'phf'}
أنواع R2L بالـ Test: {'ftp_write', 'imap', 'multihop', 'warezmaster', 'guess_passwd', 'phf'}

أنواع موجودة بالـ Test وغير موجودة بالـ Train: set()

عدد صفوف R2L من أنواع جديدة كليًا: 0 من أصل 2199


In [11]:
# نشوف توزيع كل نوع R2L بالتفصيل بكل من Train و Test
print("--- توزيع أنواع R2L بالـ Train ---")
print(df_train[df_train['attack_category']=='R2L']['label'].value_counts())

print("\n--- توزيع أنواع R2L بالـ Test ---")
print(df_test[df_test['attack_category']=='R2L']['label'].value_counts())

--- توزيع أنواع R2L بالـ Train ---
label
warezclient     890
guess_passwd     53
warezmaster      20
imap             11
ftp_write         8
multihop          7
phf               4
spy               2
Name: count, dtype: int64

--- توزيع أنواع R2L بالـ Test ---
label
guess_passwd    1231
warezmaster      944
multihop          18
ftp_write          3
phf                2
imap               1
Name: count, dtype: int64


In [12]:
from sklearn.tree import DecisionTreeClassifier

pipeline_tree = ImbPipeline([
    ("smotenc", SMOTENC(categorical_features=categorical_features_idx, random_state=42, k_neighbors=5)),
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42, max_depth=20))  # max_depth عشان نمنع Overfitting الشديد
])

pipeline_tree.fit(X_train, y_train)
y_pred_tree = pipeline_tree.predict(X_test)

print(f"Accuracy (Decision Tree): {accuracy_score(y_test, y_pred_tree):.4f}")
print(classification_report(y_test, y_pred_tree, zero_division=0))

c:\Users\Baraa Hereh\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\Baraa Hereh\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Accuracy (Decision Tree): 0.7125
              precision    recall  f1-score   support

         DoS       0.88      0.95      0.91      5741
       Probe       0.36      0.99      0.52      1106
         R2L       0.89      0.09      0.16      2199
         U2R       0.14      0.32      0.20        37
     Unknown       0.00      0.00      0.00      3750
      normal       0.72      0.96      0.82      9711

    accuracy                           0.71     22544
   macro avg       0.50      0.55      0.44     22544
weighted avg       0.64      0.71      0.63     22544

